In [1]:
#import thư viện
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

d:\ai-service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/Amazon-Products_processed.csv")

index = faiss.read_index("../models/product_index.faiss")

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2147.76it/s]


In [3]:
def recommend_products(query, top_k=5):
    # Tạo embedding cho câu truy vấn
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Chuẩn hóa
    faiss.normalize_L2(query_embedding)

    # Tìm kiếm
    scores, indices = index.search(query_embedding, top_k)

    # Lấy kết quả
    recommendations = df.iloc[indices[0]].copy()

    # Thêm điểm tương đồng
    recommendations["similarity"] = scores[0]

    return recommendations

In [4]:
recommend_products("iphone 15", top_k=5)

,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price,actual_price_vnd,discount_price_vnd,discount_percent,price_range,similarity
11326,SupCares Edge to Edge Privacy Tempered Glass f...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/61uB9JKZb2...,https://www.amazon.in/SupCares-Privacy-Tempere...,4.2,282,399.0,799.0,219964.7,109844.7,50.06,Low,0.584036
14465,iPhone Original 20W Super Fast PD Smart Charge...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/51QLU+bV6v...,https://www.amazon.in/Original-Charger-Lightin...,3.3,7,839.0,1999.0,550324.7,230976.7,58.03,Medium,0.571510
169767,Wayona MFI Certified USB C to Lightning Cable ...,"tv, audio & cameras",Home Entertainment Systems,https://m.media-amazon.com/images/I/51oX2SZd6D...,https://www.amazon.in/Wayona-Certified-Lightni...,4.1,1214,699.0,1199.0,330084.7,192434.7,41.70,Low,0.567848
18237,Apple iPhone 13 (256GB) - Midnight,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/61VuVU94Rn...,https://www.amazon.in/Apple-iPhone-13-256GB-Mi...,4.6,13932,71999.0,79900.0,21996470.0,19821324.7,9.89,Premium,0.566880
14481,iPhone Original 20W PD Adapter Compatible for ...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/iPhone-Original-Adapter-...,5.0,3,475.0,1499.0,412674.7,130767.5,68.31,Low,0.566661


In [5]:
recommend_products("wireless bluetooth headphones", top_k=5)

,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price,actual_price_vnd,discount_price_vnd,discount_percent,price_range,similarity
155574,B&O Play 1644126 E8 Truly Wireless Bluetooth i...,"tv, audio & cameras",Headphones,https://m.media-amazon.com/images/I/61bP8foeft...,https://www.amazon.in/Play-1644126-Bluetooth-H...,3.1,9,24687.81,39500.5,10874487.65,6796554.093,37.50,Premium,0.691076
155391,Go Offer Bluetooth Truly Wireless In Ear Earbu...,"tv, audio & cameras",Headphones,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/EROXLYNN-Wireless-Blueto...,3.0,93,159.00,999.0,275024.70,43772.700,84.08,Low,0.689931
153740,Sony Wh-Ch510 Bluetooth Wireless On Ear Headph...,"tv, audio & cameras",Headphones,https://m.media-amazon.com/images/I/51i+LdztEB...,https://www.amazon.in/SONY-WH-CH510-Wireless-H...,4.0,3224,2299.00,4990.0,1373747.00,632914.700,53.93,Medium,0.687771
159572,WORRICOW Bluetooth Wireless in Ear Earphones w...,"tv, audio & cameras",Headphones,https://m.media-amazon.com/images/I/51AlydCKN1...,https://www.amazon.in/WORRICOW-Bluetooth-Wirel...,2.0,2,429.00,1200.0,330360.00,118103.700,64.25,Low,0.686672
156956,Luisport Open Ear Bluetooth Headphones Wireles...,"tv, audio & cameras",Headphones,https://m.media-amazon.com/images/I/611WZtUl3W...,https://www.amazon.in/Wireless-Earbuds-Earphon...,3.7,492,2799.00,3799.0,1045864.70,770564.700,26.32,Medium,0.686068


In [6]:
embeddings = np.load("../models/embeddings.npy").astype("float32")

In [7]:
def recommend_by_product(product_index, top_k=5):
    query = embeddings[product_index].reshape(1, -1)

    scores, indices = index.search(query, top_k + 1)

    result = df.iloc[indices[0]].copy()
    result["similarity"] = scores[0]

    # Loại bỏ chính sản phẩm đang xem
    result = result[result.index != product_index]

    return result.head(top_k)

In [8]:
recommend_by_product(150)

,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price,actual_price_vnd,discount_price_vnd,discount_percent,price_range,similarity
163272,O-General 1.5 Ton 3 Star Inverter Split Air Co...,appliances,Heating & Cooling Appliances,https://m.media-amazon.com/images/I/61HcdgHcjp...,https://www.amazon.in/General-Inverter-Split-C...,3.5,11,50300.0,57900.0,15939870.0,13847590.0,13.13,Premium,0.987963
173,O-General 2.0 Ton 5 Star Inverter Split Air Co...,appliances,Air Conditioners,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/General-Star-Inverter-Sp...,3.2,7,77490.0,89470.0,24631091.0,21332997.0,13.39,Premium,0.946881
164188,O-General 2.0 Ton 5 Star Inverter Split Air Co...,appliances,Heating & Cooling Appliances,https://m.media-amazon.com/images/I/61C4vZQ5ji...,https://www.amazon.in/General-Star-Inverter-Sp...,3.2,7,77490.0,89470.0,24631091.0,21332997.0,13.39,Premium,0.937601
204,O-General 2.0 Ton 4 Star EFFICIENT & TROPICAL ...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51kxkiSq+X...,https://www.amazon.in/General-EFFICIENT-TROPIC...,3.2,2,62800.0,72630.0,19995039.0,17288840.0,13.53,Premium,0.923973
114,O-General 1.5 Ton 5 Star EFFICIENT & TROPICAL ...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51IaRZLgDE...,https://www.amazon.in/General-EFFICIENT-TROPIC...,2.7,16,61000.0,70530.0,19416909.0,16793300.0,13.51,Premium,0.918720
